# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
#from dotenv import load_dotenv, find_dotenv
import os
from google.colab import userdata
#_ = load_dotenv(find_dotenv())

#OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

### Example 1

#### Create our QandA application

In [4]:
# 1. Full clean uninstall (includes corrupted packages)
!pip uninstall -y rasterio jax jaxlib opencv-python opencv-python-headless opencv-contrib-python shap xarray-einstats pytensor tobler

!pip uninstall langchain langchain-core langchain-community langchain-openai langchain-huggingface langchain-text-splitters langchain-text-splitters-* -y

# 2. Clear pip cache
!pip cache purge

# 3. Install compatible pinned versions
!pip install "langchain-core>=0.2.28,<0.3" langchain-openai langchain-huggingface "langchain-community>=0.2.16,<0.3" langchain-text-splitters "ragas>=0.1.0"

# 4. Verify
!pip list | grep langchain

ERROR: Invalid requirement: 'langchain-text-splitters-*': Expected end or semicolon (after name and no valid version specifier)
    langchain-text-splitters-*
                            ^
Files removed: 498
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-text-splitters t

langchain                                0.2.17
langchain-classic                        1.0.1
langchain-community                      0.2.19
langchain-core                           0.2.43
langchain-huggingface                    0.0.3
langchain-openai                         0.1.25
langchain-text-splitters                 0.2.4


In [2]:
import sys
!{sys.executable} -m pip list | grep langchain

langchain                                0.2.17
langchain-classic                        1.0.1
langchain-community                      0.2.19
langchain-core                           0.2.43
langchain-huggingface                    0.0.3
langchain-openai                         0.1.25
langchain-text-splitters                 0.2.4


In [3]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain  # ✅ FIXED
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain.evaluation.qa import QAGenerateChain

In [42]:
import pandas as pd

#file = '/content/sample_data/OutdoorClothingCatalog_1000.csv'
file = '/content/sample_data/myntra_products_catalog.csv'

df = pd.read_csv(file)

df = df[["ProductName", "Description"]]

df = df.rename(columns={
    "ProductName": "name",
    "Description": "description"
    })

temp_file = "/content/sample_data/filtered.csv"
df.to_csv(temp_file, index=False)

#loader = CSVLoader(file_path=file)
loader = CSVLoader(file_path=temp_file)
data = loader.load()

In [43]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)

# Create embeddings and vectorstore
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

vectorstore = InMemoryVectorStore.from_documents(
    docs, embeddings  # embeds and indexes documents
)

retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
from langchain_core.runnables import RunnablePassthrough
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(temperature=0.0)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

<<<<>>>>>

QUESTION: {question}

ANSWER:
""")

qa = create_stuff_documents_chain(llm, prompt, document_separator="<<<<>>>>>")

# Fixed chain - provides both context AND question
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa
)


result = chain.invoke("Do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have a TSA lock?")
print(result)

Yes, the DKNY Unisex Gold & White Printed Cabin Trolley Bag does have a TSA lock.


#### Coming up with test datapoints

In [47]:
data[10]

Document(metadata={'source': '/content/sample_data/filtered.csv', 'row': 10}, page_content='name: Kenneth Cole Women Navy Blue Solid Backpack\ndescription: Navy Blue backpackNon-Padded haul loop1 main compartment with zip closurePadded backZip PocketPadded shoulder strap: PaddedWater-resistance: No')

In [48]:
data[11]

Document(metadata={'source': '/content/sample_data/filtered.csv', 'row': 11}, page_content='name: Parx Men Green Printed Polo Collar T-shirt\ndescription: Green printed T-shirt, has a polo collar, and short sleeves')

#### Hard-coded examples

In [50]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

examples = [
    {
        "query": "Do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have a TSA lock?",
        "answer": "Yes"
    },
    {
        "query": "How many pockets the Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have?",
        "answer": "Two pockets"
    }
]

prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have a TSA lock?\n"
             "   Answer: Yes\n"
             "2. Query: How many pockets the Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have?\n"
             "   Answer: Two pockets\n"
             "Query: {query}\n"
             "Answer:"
)


class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)


llm = ChatOpenAI(model="gpt-3.5-turbo")
llm_chain = prompt_template | llm | AnswerOutputParser()

query = "What colors do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have?"
result = llm_chain.invoke({"query": query})
print(result.answer)

Gold and white


In [51]:
# Run this to see what's in langchain.evaluation
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

['ContextQAEvalChain', 'CotQAEvalChain', 'QAEvalChain', 'QAGenerateChain', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'eval_chain', 'eval_prompt', 'generate_chain', 'generate_prompt']


#### LLM-Generated examples

In [52]:
from langchain.evaluation.qa import QAGenerateChain
from langchain.chains import LLMChain

example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model="gpt-3.5-turbo"))

# Your EXACT workflow now works!
llm_chain = LLMChain(llm=llm, prompt=prompt_template, output_parser=AnswerOutputParser())

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

d_flattened = [data['qa_pairs'] for data in new_examples]


/usr/local/lib/python3.12/dist-packages/langchain/chains/llm.py:366: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


#### Combine examples

In [53]:
# examples += new_example
examples += d_flattened

In [54]:
examples

[{'query': 'Do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have a TSA lock?',
  'answer': 'Yes'},
 {'query': 'How many pockets the Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have?',
  'answer': 'Two pockets'},
 {'query': 'What is the warranty period provided for the DKNY Unisex Black & Grey Printed Medium Trolley Bag?',
  'answer': 'The warranty for the DKNY Unisex Black & Grey Printed Medium Trolley Bag is 5 years.'},
 {'query': 'What does the term "Made to Measure" refer to in the description of the EthnoVogue Women Beige & Grey Kurta Set with Jacket?',
  'answer': 'The term "Made to Measure" refers to a customised kurta set according to your bust and length measurements.'},
 {'query': "What is the name of the women's jeans described in the document?",
  'answer': 'SPYKAR Women Pink Alexa Super Skinny Fit High-Rise Clean Look Stretchable Cropped Jeans'},
 {'query': 'What is the color and design of the Raymond Men Blue Self-Design Single-Breasted Bandhga

In [55]:
result = qa.invoke({
    "question": examples[6]["query"],  # ← "question" not "input"
    "context": retriever.invoke(examples[6]["query"])
})
print(result)

The color is red and black, and the pattern is printed.


### Manual Evaluation - Fun part

In [56]:
import langchain
langchain.debug = True

# qa needs BOTH question AND context
result = qa.invoke({
    "question": examples[0]["query"],
    "context": retriever.invoke(examples[0]["query"])
})
print(result)

[chain/start] [chain:stuff_documents_chain] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context>] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] Entering Chain run with input:
[inputs]
[chain/end] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] s] Exiting Chain run with output:
{
  "output": "name: DKNY Unisex Gold & White Printed Cabin Trolley Bag\ndescription: Gold and white printed cabin trolley bag, secured with a TSA lockOne handle on the top and one on the side, has a trolley with a retractable handle on the top and four corner mounted inline skate wheelsOne main zip compartment, zip lining, two compression straps with clic

In [57]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [58]:
# predictions must be list of dicts like:
# [{'query': '...', 'answer': '...', 'result': '...'}, ...]

predictions = []
for i, eg in enumerate(examples[:5]):  # Limit for testing
    docs = retriever.invoke(eg["query"])
    result = qa.invoke({"question": eg["query"], "context": docs})

    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result
    }
    predictions.append(pred_dict)

# Now your print loop works perfectly!
for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print()

Example 0:
Question: Do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have a TSA lock?
Real Answer: Yes
Predicted Answer: Yes, the DKNY Unisex Gold & White Printed Cabin Trolley Bag has a TSA lock.

Example 1:
Question: How many pockets the Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have?
Real Answer: Two pockets
Predicted Answer: The Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have two pockets.

Example 2:
Question: What is the warranty period provided for the DKNY Unisex Black & Grey Printed Medium Trolley Bag?
Real Answer: The warranty for the DKNY Unisex Black & Grey Printed Medium Trolley Bag is 5 years.
Predicted Answer: The warranty period provided for the DKNY Unisex Black & Grey Printed Medium Trolley Bag is 5 years.

Example 3:
Question: What does the term "Made to Measure" refer to in the description of the EthnoVogue Women Beige & Grey Kurta Set with Jacket?
Real Answer: The term "Made to Measure" refers to a customised kurta 

In [59]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Simple eval prompt (no classes)
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()  # Simple string parser

# Your EXACT workflow (no classes)
graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())  # Extract number

    graded_outputs.append({'score': score})

    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

Example 0:
Question: Do the DKNY Unisex Gold & White Printed Cabin Trolley Bag have a TSA lock?
Real: Yes
Predicted: Yes, the DKNY Unisex Gold & White Printed Cabin Trolley Bag has a TSA lock.
Score: 1.0 

Example 1:
Question: How many pockets the Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have?
Real: Two pockets
Predicted: The Urban Dog Men Charcoal Black Printed Regular Fit Sports Shorts have two pockets.
Score: 1.0 

Example 2:
Question: What is the warranty period provided for the DKNY Unisex Black & Grey Printed Medium Trolley Bag?
Real: The warranty for the DKNY Unisex Black & Grey Printed Medium Trolley Bag is 5 years.
Predicted: The warranty period provided for the DKNY Unisex Black & Grey Printed Medium Trolley Bag is 5 years.
Score: 1.0 

Example 3:
Question: What does the term "Made to Measure" refer to in the description of the EthnoVogue Women Beige & Grey Kurta Set with Jacket?
Real: The term "Made to Measure" refers to a customised kurta set according

In [60]:
graded_outputs

[{'score': 1.0},
 {'score': 1.0},
 {'score': 1.0},
 {'score': 1.0},
 {'score': 0.2}]

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [61]:
from langchain_community.document_loaders import TextLoader
from langchain_core.runnables import RunnablePassthrough

# Load the text file
loader = TextLoader("/content/sample_data/nyc_text.txt")
docs = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)

# Create vectorstore
vectorstore = InMemoryVectorStore.from_documents(
    split_docs, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

# Create QA chain
llm = ChatOpenAI(temperature=0)
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

qa_chain = create_stuff_documents_chain(llm, prompt)

# Combine into runnable chain that returns both answer and source documents
qa_chain_with_docs = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa_chain
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain_with_docs.invoke(question)
print(result)

New York City was named after King Charles II of England granted the lands to his brother, the Duke of York.


Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [63]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [64]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [24]:
#!pip install ragas

In [65]:

from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

all_predictions = []
for example in examples:
    q = example["query"]
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_predictions.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": example["ground_truths"][0]
    })

dataset_all = Dataset.from_list(all_predictions)

print("evaluating faithfulness...")
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

/tmp/ipykernel_10932/3903234907.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_10932/3903234907.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


evaluating faithfulness...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 0.8000}


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

**Faithfulness**

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [66]:
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

evaluating context recall...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.7333}


In [67]:
from ragas import evaluate
from ragas.metrics import faithfulness
from datasets import Dataset

# Use question 2 (economic significance) — produces long detailed answer
high_data = Dataset.from_list([all_predictions[2]])  # economic significance question
high_score = evaluate(high_data, [faithfulness])["faithfulness"]
print("HIGH Faithfulness (real answer):", high_score)

# LOW — contradicting claims
fake = all_predictions[2].copy()
fake["answer"] = "New York City has no economic importance. It is a small rural town with no financial institutions, no corporations, and no international trade. Wall Street closed in 1990."
low_data = Dataset.from_list([fake])
low_score = evaluate(low_data, [faithfulness])["faithfulness"]
print("LOW Faithfulness (fake answer):", low_score)

print("\n--- Explanation ---")
print(f"Real answer score:  {high_score[0]:.2f}  <- claims supported by context")
print(f"Fake answer score:  {low_score[0]:.2f}  <- claims contradict context")

/tmp/ipykernel_10932/252685123.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

HIGH Faithfulness (real answer): [1.0]


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

LOW Faithfulness (fake answer): [0.0]

--- Explanation ---
Real answer score:  1.00  <- claims supported by context
Fake answer score:  0.00  <- claims contradict context


In [32]:
print("Question:", all_predictions[2]["question"])
print("Answer:", all_predictions[2]["answer"])

Question: What is the economic significance of New York City?
Answer: New York City is a global hub of business and commerce, with a diverse economy that includes banking and finance, health care, technology, retail, tourism, real estate, media, advertising, legal services, and more. It is a major economic engine with a gross metropolitan product of over $2.4 trillion, making it the largest metropolitan economy in the world. Additionally, New York City is home to the highest number of billionaires, individuals of ultra-high net worth, and millionaires of any city in the world, making it an established safe haven for global investors.


**Context Relevancy**

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [68]:
from ragas.metrics import context_recall

# HIGH — real docs
high_data = Dataset.from_list([all_predictions[2]])
high_score = evaluate(high_data, [context_recall])["context_recall"]
print("HIGH Context Recall (real docs):", high_score)

# LOW — off-topic docs
fake = all_predictions[2].copy()
fake["contexts"] = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language used in data science."
]
low_data = Dataset.from_list([fake])
low_score = evaluate(low_data, [context_recall])["context_recall"]
print("LOW Context Recall (fake docs):", low_score)

print("\n--- Explanation ---")
print(f"Real docs score:  {high_score[0]:.2f}  ← ground truth found in context")
print(f"Fake docs score:  {low_score[0]:.2f}  ← ground truth absent from context")

/tmp/ipykernel_10932/2458786270.py:1: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

HIGH Context Recall (real docs): [1.0]


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

LOW Context Recall (fake docs): [0.0]

--- Explanation ---
Real docs score:  1.00  ← ground truth found in context
Fake docs score:  0.00  ← ground truth absent from context


In [69]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

# helper
def get_contexts_str(contexts):
    return [
        doc.page_content if hasattr(doc, "page_content") else str(doc)
        for doc in contexts
    ] if contexts and hasattr(contexts[0], "page_content") else contexts

def evaluate_answer(question, answer, contexts, ground_truth=None, test_name=""):
    contexts_str = get_contexts_str(contexts)

    data = {
        "question": [question],
        "answer":   [answer],
        "contexts": [contexts_str],
        "ground_truth": [ground_truth if ground_truth else answer]
    }

    dataset = Dataset.from_dict(data)
    results = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy, context_recall])

    print(f"\n{'='*60}")
    print(f"TEST: {test_name}")
    print(f"Faithfulness:     {results['faithfulness'][0]:.4f}")
    print(f"Answer Relevancy: {results['answer_relevancy'][0]:.4f}")
    print(f"Context Recall:   {results['context_recall'][0]:.4f}")
    return results

# Test it
evaluate_answer(
    question=all_predictions[2]["question"],
    answer=all_predictions[2]["answer"],
    contexts=all_predictions[2]["contexts"],
    ground_truth=all_predictions[2]["ground_truth"],
    test_name="Economic Significance"
)

/tmp/ipykernel_10932/1871168344.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
/tmp/ipykernel_10932/1871168344.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
/tmp/ipykernel_10932/1871168344.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, answer_relevancy, context_recall


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')



TEST: Economic Significance
Faithfulness:     1.0000
Answer Relevancy: nan
Context Recall:   1.0000


{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_recall': 1.0000}

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [70]:
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

dataset_all = Dataset.from_list(all_predictions)

results = evaluate(
    dataset_all,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print(results)
results.to_pandas()

/tmp/ipykernel_10932/3324676083.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_10932/3324676083.py:1: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_10932/3324676083.py:1: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfu

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[9]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[13]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[17]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')


{'faithfulness': 0.8000, 'answer_relevancy': nan, 'context_precision': 0.8500, 'context_recall': 0.7333}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the population of New York City as of ...,[New York City is the most populous city in th...,The population of New York City as of 2020 is ...,"8,804,190",1.0,NaN,1.00,1.000000
1,Which borough of New York City has the highest...,[New York City is the most populous city in th...,Manhattan,Brooklyn,0.0,NaN,0.75,0.000000
2,What is the economic significance of New York ...,[Despite New York's heavy reliance on its vast...,New York City is a global hub of business and ...,"New York City's economic significance is vast,...",1.0,NaN,0.50,1.000000
3,How did New York City get its name?,[The city and its metropolitan area constitute...,New York City was named after King Charles II ...,New York City got its name when it came under ...,1.0,NaN,1.00,1.000000
4,What is the significance of the Statue of Libe...,[the Dutch in July 1673 and was renamed New Or...,The Statue of Liberty is a symbol of the U.S. ...,The Statue of Liberty in New York City holds g...,1.0,NaN,1.00,0.666667


In [71]:
# run the queries as a batch for efficiency
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

# Step 1 — batch predictions (same idea as qa_chain.batch(examples))
all_preds = []
for example in examples:
    q = example["query"]
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_preds.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": example["ground_truths"][0]
    })

dataset_all = Dataset.from_list(all_preds)

# Step 2 — replaces faithfulness_chain.evaluate(examples, predictions)
print("evaluating faithfulness...")
r = evaluate(dataset_all, metrics=[faithfulness])
print(r)

# Step 3 — replaces context_recall_chain.evaluate(examples, predictions)
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

/tmp/ipykernel_10932/57311970.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_10932/57311970.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


evaluating faithfulness...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 0.9333}
evaluating context recall...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.7333}
